# Render Annotated Video

This notebook loads video files and burns the labels from `data/annotations.csv` onto the frames, saving it out as a new `.mp4` video.

This is the fastest way to review annotations as you can just watch the video natively and spot-check for any missed punches or mislabeled ones!

In [1]:
import cv2
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

# Settings
VIDEO_NAME = 'V1' 
INPUT_VIDEO_PATH = Path(f'../data/downloaded-videos/{VIDEO_NAME}.mp4')
ANNOTATIONS_PATH = Path('../data/annotations.csv')
OUTPUT_VIDEO_PATH = Path(f'../data/{VIDEO_NAME}_annotated.mp4')

In [2]:
# 1. Load annotations
# Inferred columns based on CSV structure
col_names = ['video_file', 'frame_number', 'label', 'flag1', 'flag2', 'timestamp']
df = pd.read_csv(ANNOTATIONS_PATH, names=col_names)

# Filter for the target video
df_video = df[df['video_file'] == f'{VIDEO_NAME}.mp4'].copy()

# Create a mapping of frame_number -> list of labels 
# (in case there are multiple punches on the exact same frame)
annotations_map = df_video.groupby('frame_number')['label'].apply(list).to_dict()

print(f"Loaded {len(df_video)} annotations for {VIDEO_NAME}")

Loaded 735 annotations for V1


In [3]:
# 2. Render a "Slideshow" video of ONLY the annotated frames
cap = cv2.VideoCapture(str(INPUT_VIDEO_PATH))

if not cap.isOpened():
    raise Exception(f"Failed to open video at {INPUT_VIDEO_PATH}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# We will create a new video that acts as a slideshow
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter(str(OUTPUT_VIDEO_PATH), fourcc, fps, (width, height))

annotated_frames = sorted(list(annotations_map.keys()))
print(f"Rendering {len(annotated_frames)} annotated frames to {OUTPUT_VIDEO_PATH} ...")

with tqdm(total=len(annotated_frames)) as pbar:
    for f_idx in annotated_frames:
        # Seek directly to the annotated frame (cast to int to avoid OpenCV TypeError)
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(f_idx))
        ret, frame = cap.read()
        
        if not ret:
            print(f"Could not read frame {f_idx}")
            pbar.update(1)
            continue
            
        labels = annotations_map[f_idx]
        label_text = f"Frame {int(f_idx)}: " + ", ".join(labels)
        
        # Position: Top-left corner
        org = (50, 100)
        font = cv2.FONT_HERSHEY_SIMPLEX
        fontScale = 2
        color = (0, 0, 255) # Red text in BGR
        thickness = 5
        
        # Draw black background outline for better visibility
        frame = cv2.putText(frame, label_text, org, font, fontScale, (0,0,0), thickness + 3, cv2.LINE_AA)
        frame = cv2.putText(frame, label_text, org, font, fontScale, color, thickness, cv2.LINE_AA)

        # Write this frame multiple times so it stays on screen for ~1.5 seconds
        # so you have time to see it and the label
        hold_frames = int(fps * 1.5)
        for _ in range(hold_frames):
            out.write(frame)
            
        pbar.update(1)

cap.release()
out.release()
print("Done! The video now exactly contains ONLY the annotated frames shown for 1.5s each.")

Rendering 725 annotated frames to ../data/V1_annotated.mp4 ...


  0%|          | 0/725 [00:00<?, ?it/s]

Done! The video now exactly contains ONLY the annotated frames shown for 1.5s each.
